# Ingesta de empleados Banco Andino
**Objetivo:** Transformar y cargar los datos de la hoja `Empleados` del archivo Excel en las tablas `persona`, `persona_documento`, `credencial_biostar` y `empleado_sede_acceso` de una base de datos PostgreSQL (Supabase).

Se aplican las reglas de limpieza y validación definidas en el análisis previo.

In [ ]:
# Celda 1: Instalación de dependencias
!pip install pandas openpyxl sqlalchemy psycopg2-binary python-dateutil

In [ ]:
# Celda 2: Importaciones
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError, IntegrityError
import uuid
from datetime import datetime, date
import re
from dateutil import parser
import getpass
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente.")

In [ ]:
# Celda 3: Conexión a la base de datos (Supabase)
# Se pide al usuario que ingrese las credenciales de forma segura
print("Ingresa los datos de conexión a tu base de datos PostgreSQL (Supabase):")
DB_HOST = input("Host (ej: db.xxxxxx.supabase.co): ")
DB_PORT = input("Puerto (default 5432): ") or "5432"
DB_NAME = input("Nombre de la base de datos (ej: postgres): ")
DB_USER = input("Usuario: ")
DB_PASSWORD = getpass.getpass("Contraseña: ")

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# Probar conexión
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
    print("✅ Conexión exitosa a la base de datos.")
except Exception as e:
    print("❌ Error de conexión:", e)
    raise

In [ ]:
# Celda 4: Carga del archivo Excel
from google.colab import files

print("Por favor, sube el archivo '03-empleados-banco-andino.xlsx'.")
uploaded = files.upload()

# Tomar el nombre del primer archivo subido
file_name = list(uploaded.keys())[0]
print(f"Archivo cargado: {file_name}")

# Leer ambas hojas
df_sedes = pd.read_excel(file_name, sheet_name="Sedes")
df_empleados = pd.read_excel(file_name, sheet_name="Empleados")

print(f"Sedes: {df_sedes.shape[0]} registros")
print(f"Empleados: {df_empleados.shape[0]} registros")
df_empleados.head()

In [ ]:
# Celda 5: Definición de funciones de limpieza y normalización

# Diccionarios de mapeo (proporcionados)
MAP_SEDE = {
    'CAL-01': '0105540b-0697-47f7-a844-8c5096c2507d',
    'BOG-CD': '0d6e45ae-f6c8-477d-b288-6bef50e4ef9d',
    'MED-01': '0e93ddf6-7a1d-4fab-9f49-547dc5e3ee7f',
    'PTY-01': '15dccbb8-0499-4b2d-bf87-4d35013c550b',
    'CHI-01': '63f7f0f6-f328-4b50-acf3-bc2584cf933f',
    'BOG-TOR': 'a0d15094-852d-4843-a638-1d2688230ccf'
}

MAP_TIPO_DOC = {
    'CC': '519013a1-9ebf-4c93-a7ae-52e9002ac873',
    'CE': '44299063-da8d-4d84-b8f8-9c73a59a5a7a',
    'CIP': '85a20deb-94ca-437a-86ce-65889d5cf3dc',
    'PAS': '6c77a7d2-753f-4bec-bf07-70601d0f4281',
    'PA': '6c77a7d2-753f-4bec-bf07-70601d0f4281',
    'N': 'f7723aef-47df-4f0c-8eec-57cef6c7bfbc'
}

# Normalización de valores semánticos
def normalizar_pais(valor):
    if pd.isna(valor) or valor is None:
        return None
    v = str(valor).strip().upper()
    if v in ['COLOMBIA', 'CO']:
        return 'Colombia'
    if v in ['PANAMÁ', 'PANAMA', 'PA']:
        return 'Panamá'
    return None  # valor no reconocido

def normalizar_estado(valor):
    if pd.isna(valor) or valor is None:
        return None
    v = str(valor).strip().upper()
    if v in ['ACTIVO', 'ACTIVE', 'A']:
        return 'Activo'
    if v in ['INACTIVO', 'INACTIVE', 'RETIRADO', 'RETIRED']:
        return 'Inactivo'
    return None

def normalizar_tipo_contrato(valor):
    if pd.isna(valor) or valor is None:
        return None
    v = str(valor).strip().upper()
    if v in ['INDEFINIDO', 'INDEF']:
        return 'Indefinido'
    if v == 'FIJO':
        return 'Fijo'
    if v == 'APRENDIZAJE':
        return 'Aprendizaje'
    return v  # se conserva tal cual, pero puede ser no válido

def parsear_fecha(valor):
    """Intenta parsear una fecha en múltiples formatos."""
    if pd.isna(valor) or valor is None or str(valor).strip() == '':
        return None
    v = str(valor).strip()
    # Reemplazar separadores comunes
    v = v.replace('/', '-')
    # Intentar con dateutil
    try:
        dt = parser.parse(v, fuzzy=True, dayfirst=False)
        return dt.date()
    except:
        return None

def limpiar_numero_documento(valor):
    """Limpia el número de documento: elimina espacios, puntos, comas.
       Si es notación científica (1.02E+09), retorna None (irrecuperable)."""
    if pd.isna(valor) or valor is None:
        return None
    v = str(valor).strip()
    # Si contiene 'E' o 'e' y dígitos, puede ser notación científica
    if re.search(r'[Ee]\+?\d+$', v):
        return None
    # Eliminar todos los puntos, comas y espacios
    v = re.sub(r'[.,\s]', '', v)
    if v == '':
        return None
    return v

def validar_email(valor):
    if pd.isna(valor) or valor is None:
        return None
    v = str(valor).strip()
    if '@' in v and '.' in v.split('@')[-1]:
        return v.lower()
    return None  # email inválido

def limpiar_telefono(valor):
    if pd.isna(valor) or valor is None:
        return None
    v = str(valor).strip()
    # Placeholder identificado
    if v == '3105551234':
        return None
    # Limpiar espacios y guiones
    v = re.sub(r'[\s\-]', '', v)
    if v == '':
        return None
    return v

def mapear_sede(codigo):
    if pd.isna(codigo) or codigo is None:
        return None
    v = str(codigo).strip().upper()
    return MAP_SEDE.get(v)

def mapear_tipo_documento(tipo):
    if pd.isna(tipo) or tipo is None:
        return None
    v = str(tipo).strip().upper()
    return MAP_TIPO_DOC.get(v)

In [ ]:
# Celda 6: Funciones de validación de duplicados y registro de errores

# Conjuntos para detectar duplicados dentro del mismo lote
documentos_insertados = set()      # (tipo_doc_uuid, numero_doc)
biostar_insertados = set()         # id_biostar
rfid_insertados = set()            # tarjeta_rfid

# Estructura para almacenar reportes
reporte = []

def registrar_reporte(codigo_empleado, motivo, accion, detalles=None):
    reporte.append({
        'codigo_empleado': codigo_empleado,
        'motivo': motivo,
        'accion_tomada': accion,
        'detalles': detalles
    })

def es_documento_duplicado(tipo_doc_uuid, numero_doc):
    clave = (tipo_doc_uuid, numero_doc)
    if clave in documentos_insertados:
        return True
    # También podríamos verificar en la base de datos, pero asumimos que está vacía
    # o que solo se ejecuta una vez. Para seguridad, se puede consultar.
    # Aquí solo usamos memoria.
    return False

def marcar_documento_insertado(tipo_doc_uuid, numero_doc):
    documentos_insertados.add((tipo_doc_uuid, numero_doc))

In [ ]:
# Celda 7: Procesamiento de empleados e inserción

# Preparar listas para inserción por lotes (opcional, pero usaremos transacción por empleado)
# Para simplificar, insertamos cada empleado en una transacción.

def procesar_fila(row):
    """Procesa una fila del DataFrame de empleados y retorna un diccionario con los datos a insertar,
       o None si la persona no cumple requisitos mínimos."""
    codigo = row['codigo_empleado']
    
    # --- 1. Validar campos requeridos de PERSONA ---
    primer_nombre = str(row['primer_nombre']).strip() if pd.notna(row['primer_nombre']) else ''
    primer_apellido = str(row['primer_apellido']).strip() if pd.notna(row['primer_apellido']) else ''
    if not primer_nombre or not primer_apellido:
        registrar_reporte(codigo, 'Campos requeridos de persona incompletos', 'Rechazado', 
                          f"Falta primer_nombre o primer_apellido: '{primer_nombre}' / '{primer_apellido}'")
        return None
    
    # --- 2. Normalizar estado y fechas ---
    estado_norm = normalizar_estado(row['estado'])
    if estado_norm is None:
        registrar_reporte(codigo, 'Estado no reconocido', 'Revisión manual', f"Valor: {row['estado']}")
        # Decidimos no insertar, porque activo es required.
        return None
    activo = (estado_norm == 'Activo')
    
    fecha_ingreso = parsear_fecha(row['fecha_ingreso'])
    fecha_retiro = parsear_fecha(row['fecha_retiro']) if pd.notna(row['fecha_retiro']) else None
    
    # Validar consistencia de fechas
    hoy = date.today()
    if fecha_ingreso is None:
        registrar_reporte(codigo, 'Fecha de ingreso inválida o ausente', 'Revisión manual', 
                          f"Valor: {row['fecha_ingreso']}")
        return None
    if fecha_ingreso > hoy:
        registrar_reporte(codigo, 'Fecha de ingreso futura', 'Revisión manual', 
                          f"Fecha: {fecha_ingreso}")
        return None
    if fecha_retiro is not None and fecha_retiro < fecha_ingreso:
        registrar_reporte(codigo, 'Fecha de retiro anterior a fecha de ingreso', 'Revisión manual', 
                          f"Ingreso: {fecha_ingreso}, Retiro: {fecha_retiro}")
        return None
    
    # Verificar consistencia estado vs fecha_retiro
    if activo and fecha_retiro is not None:
        registrar_reporte(codigo, 'Estado activo con fecha de retiro', 'Advertencia', 
                          f"Se ignora fecha de retiro: {fecha_retiro}")
        fecha_retiro = None  # forzamos NULL para activos
    if not activo and fecha_retiro is None:
        registrar_reporte(codigo, 'Estado inactivo sin fecha de retiro', 'Advertencia', 
                          'Se asigna fecha de retiro NULL')
        # No se puede adivinar, dejamos NULL
    
    # --- 3. Otros campos ---
    pais = normalizar_pais(row['pais'])
    # Nota: pais no se almacena en persona en el ER, pero podríamos usarlo para validar sede.
    
    segundo_nombre = str(row['segundo_nombre']).strip() if pd.notna(row['segundo_nombre']) else None
    segundo_apellido = str(row['segundo_apellido']).strip() if pd.notna(row['segundo_apellido']) else None
    telefono = limpiar_telefono(row['telefono'])
    email = validar_email(row['email'])
    if email is None and pd.notna(row['email']):
        registrar_reporte(codigo, 'Email inválido', 'Advertencia', f"Valor: {row['email']}")
    
    nivel_acceso = str(row['nivel_acceso']).strip() if pd.notna(row['nivel_acceso']) else None
    jornada = str(row['jornada']).strip() if pd.notna(row['jornada']) else None
    cargo = str(row['cargo']).strip() if pd.notna(row['cargo']) else None
    area = str(row['area']).strip() if pd.notna(row['area']) else None
    centro_costo = str(row['centro_costo']).strip() if pd.notna(row['centro_costo']) else None
    tipo_contrato = normalizar_tipo_contrato(row['tipo_contrato'])
    
    # --- 4. Mapear sede ---
    sede_id = mapear_sede(row['sede'])
    if sede_id is None:
        registrar_reporte(codigo, 'Sede no mapeable o inexistente', 'Revisión manual', 
                          f"Valor: {row['sede']}")
        # La persona se insertará, pero no se creará registro en empleado_sede_acceso.
    
    # --- 5. Mapear tipo documento ---
    tipo_doc = row['tipo_doc']
    tipo_doc_uuid = mapear_tipo_documento(tipo_doc)
    numero_doc = limpiar_numero_documento(row['numero_documento'])
    documento_valido = (tipo_doc_uuid is not None and numero_doc is not None)
    if not documento_valido:
        registrar_reporte(codigo, 'Documento inválido o no mapeable', 'Revisión manual', 
                          f"tipo: {tipo_doc}, número: {row['numero_documento']}")
    
    # --- 6. Credencial Biostar ---
    id_biostar = str(row['id_biostar']).strip() if pd.notna(row['id_biostar']) else None
    tarjeta_rfid = str(row['tarjeta_rfid']).strip() if pd.notna(row['tarjeta_rfid']) else None
    credencial_valida = (id_biostar is not None and id_biostar != '' and 
                         tarjeta_rfid is not None and tarjeta_rfid != '')
    if not credencial_valida:
        registrar_reporte(codigo, 'Credencial incompleta (falta id_biostar o tarjeta_rfid)', 'Advertencia', 
                          f"id_biostar: {id_biostar}, tarjeta_rfid: {tarjeta_rfid}")
    
    # --- Construir diccionario para PERSONA ---
    persona_id = uuid.uuid4()
    persona_data = {
        'id': persona_id,
        'tipo_persona': 'EMPLEADO',
        'primer_nombre': primer_nombre,
        'segundo_nombre': segundo_nombre,
        'primer_apellido': primer_apellido,
        'segundo_apellido': segundo_apellido,
        'telefono': telefono,
        'email': email,
        'codigo_empleado': str(codigo) if pd.notna(codigo) else None,
        'fecha_ingreso': fecha_ingreso,
        'fecha_retiro': fecha_retiro,
        'activo': activo,
        'nivel_acceso': nivel_acceso,
        'jornada': jornada,
        'cargo': cargo,
        'area': area,
        'centro_costo': centro_costo,
        'tipo_contrato': tipo_contrato
    }
    
    # Preparar dependencias
    dependencias = []
    
    # Documento
    if documento_valido:
        # Verificar duplicado en el lote
        if es_documento_duplicado(tipo_doc_uuid, numero_doc):
            registrar_reporte(codigo, 'Documento duplicado en el lote', 'Revisión manual',
                              f"tipo: {tipo_doc}, número: {numero_doc}")
        else:
            doc_data = {
                'id_documento': tipo_doc_uuid,
                'numero_documento': numero_doc,
                'id_persona': persona_id,
                'es_principal': True
            }
            dependencias.append(('persona_documento', doc_data))
            marcar_documento_insertado(tipo_doc_uuid, numero_doc)
    
    # Credencial
    if credencial_valida:
        # Verificar duplicados de id_biostar y tarjeta_rfid
        if id_biostar in biostar_insertados:
            registrar_reporte(codigo, 'id_biostar duplicado', 'Revisión manual',
                              f"id_biostar: {id_biostar}")
        elif tarjeta_rfid in rfid_insertados:
            registrar_reporte(codigo, 'tarjeta_rfid duplicada', 'Revisión manual',
                              f"tarjeta_rfid: {tarjeta_rfid}")
        else:
            cred_data = {
                'id': uuid.uuid4(),
                'empleado_id': persona_id,
                'id_biostar': id_biostar,
                'tarjeta_rfid': tarjeta_rfid,
                'esta_sincronizado': False  # por defecto
            }
            dependencias.append(('credencial_biostar', cred_data))
            biostar_insertados.add(id_biostar)
            rfid_insertados.add(tarjeta_rfid)
    
    # Sede acceso (solo si sede_id no es None)
    if sede_id is not None:
        acceso_data = {
            'empleado_id': persona_id,
            'sede_id': sede_id,
            'es_principal': True
        }
        dependencias.append(('empleado_sede_acceso', acceso_data))
    
    return {
        'persona': persona_data,
        'dependencias': dependencias
    }


# Función para insertar en base de datos con transacción
def insertar_empleado(engine, data):
    """Inserta la persona y sus dependencias en una transacción."""
    if data is None:
        return
    persona = data['persona']
    dependencias = data['dependencias']
    
    with engine.connect() as conn:
        trans = conn.begin()
        try:
            # Insertar persona
            insert_persona = text("""
                INSERT INTO persona (
                    id, tipo_persona, primer_nombre, segundo_nombre,
                    primer_apellido, segundo_apellido, telefono, email,
                    codigo_empleado, fecha_ingreso, fecha_retiro, activo,
                    nivel_acceso, jornada, cargo, area, centro_costo, tipo_contrato
                ) VALUES (
                    :id, :tipo_persona, :primer_nombre, :segundo_nombre,
                    :primer_apellido, :segundo_apellido, :telefono, :email,
                    :codigo_empleado, :fecha_ingreso, :fecha_retiro, :activo,
                    :nivel_acceso, :jornada, :cargo, :area, :centro_costo, :tipo_contrato
                )
            """)
            conn.execute(insert_persona, persona)
            
            # Insertar dependencias
            for tabla, datos in dependencias:
                if tabla == 'persona_documento':
                    sql = text("""
                        INSERT INTO persona_documento (id_documento, numero_documento, id_persona, es_principal)
                        VALUES (:id_documento, :numero_documento, :id_persona, :es_principal)
                    """)
                elif tabla == 'credencial_biostar':
                    sql = text("""
                        INSERT INTO credencial_biostar (id, empleado_id, id_biostar, tarjeta_rfid, esta_sincronizado)
                        VALUES (:id, :empleado_id, :id_biostar, :tarjeta_rfid, :esta_sincronizado)
                    """)
                elif tabla == 'empleado_sede_acceso':
                    sql = text("""
                        INSERT INTO empleado_sede_acceso (empleado_id, sede_id, es_principal)
                        VALUES (:empleado_id, :sede_id, :es_principal)
                    """)
                else:
                    continue
                conn.execute(sql, datos)
            
            trans.commit()
            return True
        except IntegrityError as e:
            trans.rollback()
            # Si hay violación de unicidad, probablemente duplicado en la base de datos
            # Registramos el error
            codigo = persona.get('codigo_empleado', 'desconocido')
            registrar_reporte(codigo, 'Error de integridad al insertar', 'Rechazado', str(e.orig))
            return False
        except Exception as e:
            trans.rollback()
            codigo = persona.get('codigo_empleado', 'desconocido')
            registrar_reporte(codigo, 'Error inesperado', 'Rechazado', str(e))
            return False

In [ ]:
# Celda 8: Ejecutar el procesamiento para todos los empleados

print("Iniciando procesamiento...")
contador = 0
total = len(df_empleados)

for idx, row in df_empleados.iterrows():
    contador += 1
    if contador % 50 == 0:
        print(f"Procesados {contador}/{total}")
    
    data = procesar_fila(row)
    if data is not None:
        # Intentar insertar
        exito = insertar_empleado(engine, data)
        if not exito:
            # Ya se registró en reporte dentro de la función de inserción
            pass
    # Si data es None, ya se registró el motivo en reporte

print(f"Procesamiento finalizado. Total procesados: {contador}")

# Generar resumen del reporte
df_reporte = pd.DataFrame(reporte)
print("\nResumen de reporte:")
print(df_reporte['accion_tomada'].value_counts().to_string())

In [ ]:
# Celda 9: Mostrar detalle de registros rechazados o en revisión

if len(df_reporte) > 0:
    print("\nRegistros con incidencias:")
    display(df_reporte[df_reporte['accion_tomada'] != 'Advertencia'])
    print("\nAdvertencias (no bloqueantes):")
    display(df_reporte[df_reporte['accion_tomada'] == 'Advertencia'])
else:
    print("Todos los registros se procesaron sin incidencias.")

# Opcional: guardar el reporte en un archivo CSV
df_reporte.to_csv('reporte_ingesta.csv', index=False, encoding='utf-8')
print("\nReporte guardado como 'reporte_ingesta.csv'")